In [28]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/karttikjangid05/config-file/config.yaml


In [29]:
!pip install -q "transformers>=4.53.0" torchcodec

In [30]:
import yaml, torch
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoVideoProcessor, AutoModel
from torchcodec.decoders import VideoDecoder

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [31]:
%%writefile config.yaml
model:
  checkpoint: facebook/vjepa2-vitl-fpc64-256
  dino_checkpoint: facebook/dinov2-base
  pooling: mean         
  secondary_pooling: max  

data:
  repo_id: nateraw/kinetics-mini
  repo_type: dataset
  revision: 9f4ed38128a355c352527209101be3e326471816
  split: val
  actions:
    - archery
    - bowling
    - high_jump
  selection: all
  num_frames: 64

perturbation:
  bootstrap_seed: 21
  bootstrap_n: 10000
  shuffle_seed: 42

Overwriting config.yaml


In [32]:
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
print(cfg)

{'model': {'checkpoint': 'facebook/vjepa2-vitl-fpc64-256', 'dino_checkpoint': 'facebook/dinov2-base', 'pooling': 'mean', 'secondary_pooling': 'max'}, 'data': {'repo_id': 'nateraw/kinetics-mini', 'repo_type': 'dataset', 'revision': '9f4ed38128a355c352527209101be3e326471816', 'split': 'val', 'actions': ['archery', 'bowling', 'high_jump'], 'selection': 'all', 'num_frames': 64}, 'perturbation': {'bootstrap_seed': 21, 'bootstrap_n': 10000, 'shuffle_seed': 42}}


In [33]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HUGGING_FACE_TOKEN")

DATA_ROOT = Path(snapshot_download(
    repo_id=cfg["data"]["repo_id"],
    repo_type=cfg["data"]["repo_type"],
    revision=cfg["data"]["revision"],
    allow_patterns=[f"{cfg['data']['split']}/{a}/*" for a in cfg["data"]["actions"]],
    token=token,
))

clip_paths = sorted(DATA_ROOT.rglob("*.mp4"))
print(len(clip_paths))

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

30


In [34]:
from transformers import AutoVideoProcessor, AutoModel , AutoProcessor
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HUGGING_FACE_TOKEN")

vjepa_model = AutoModel.from_pretrained(cfg["model"]["checkpoint"], token=token)
vjepa_processor = AutoVideoProcessor.from_pretrained(cfg["model"]["checkpoint"], token=token)
vjepa_model.eval()

vjepa_model.to("cuda")

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

VJEPA2Model(
  (encoder): VJEPA2Encoder(
    (embeddings): VJEPA2Embeddings(
      (patch_embeddings): VJEPA2PatchEmbeddings3D(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-23): 24 x VJEPA2Layer(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA2RopeAttention(
          (query): Linear(in_features=1024, out_features=1024, bias=True)
          (key): Linear(in_features=1024, out_features=1024, bias=True)
          (value): Linear(in_features=1024, out_features=1024, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): VJEPA2MLP(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (activation): GELUActivation()
          (fc

In [35]:
from torchcodec.decoders import VideoDecoder
import time

## checking for time
torch.cuda.synchronize()
start_time = time.time()
required_frames = cfg["data"]["num_frames"]
reference_video_url = clip_paths[0]
decoder = VideoDecoder(reference_video_url, device="cuda")


frame_indices = list(range(0 , required_frames))

raw_video_tensor = decoder.get_frames_at(indices = frame_indices).data

print(decoder.metadata)

print(raw_video_tensor.shape, raw_video_tensor.dtype)

inputs = vjepa_processor(raw_video_tensor ,return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = vjepa_model(**inputs)
    features = outputs.last_hidden_state

torch.cuda.synchronize()
end_time = time.time()

clip_duration = end_time - start_time

for p in clip_paths:
    n = VideoDecoder(p).metadata.num_frames
    if n < cfg["data"]["num_frames"]:
        print("TOO SHORT:", p.name, n)
print("checked", len(clip_paths))
print(f"the GPU time required for each clip is {clip_duration}")
print(f"Pipeline verified! Extracted features shape: {features.shape}")

VideoStreamMetadata:
  duration_seconds_from_header: 10.01
  begin_stream_seconds_from_header: 0.0
  bit_rate: 850570.0
  codec: h264
  stream_index: 0
  duration_seconds: 10.01
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: 0.0
  end_stream_seconds_from_content: 10.01
  width: 480
  height: 360
  num_frames_from_header: 300
  num_frames_from_content: 300
  average_fps_from_header: 29.97003
  pixel_aspect_ratio: 1
  end_stream_seconds: 10.01
  num_frames: 300
  average_fps: 29.97003

torch.Size([64, 3, 360, 480]) torch.uint8
checked 30
the GPU time required for each clip is 6.7240376472473145
Pipeline verified! Extracted features shape: torch.Size([1, 8192, 1024])


In [36]:
def reverse_clip(raw_video_tensor):
    flipped_tensor = torch.flip(raw_video_tensor , dims =[0])
    return flipped_tensor
    

In [37]:
def shuffle_clip(frames: torch.Tensor, seed: int) -> torch.Tensor:
    """
    Shuffles frames along the temporal dimension (T, C, H, W) deterministically.
    """
    # 1. Inspect the time dimension length (T)
    num_frames = frames.shape[0]

    # 2. Set up an isolated CPU generator for reproducibility
    generator = torch.Generator().manual_seed(seed)

    # 3. Generate deterministic permuted indices [0, num_frames - 1]
    permuted_indices = torch.randperm(num_frames, generator=generator)

    # Ensure indices reside on the same device as the video frames (e.g., CUDA)
    permuted_indices = permuted_indices.to(frames.device)

    # 4. Reorder along dimension 0 via tensor indexing
    shuffled_frames = frames[permuted_indices]

    return shuffled_frames

In [38]:
def make_static_clip(frames: torch.Tensor) -> torch.Tensor:
    """
    Takes the middle frame and repeats it T times along the time axis,
    preserving the original (T, C, H, W) shape.
    """
    T = frames.shape[0]
    mid_idx = T // 2
    
    # Extract middle frame while keeping dimension 0: shape becomes (1, C, H, W)
    mid_frame = frames[mid_idx : mid_idx + 1]
    
    # Repeat along the time dimension T times without altering C, H, W
    static_frames = mid_frame.repeat(T, 1, 1, 1)
    
    return static_frames

In [39]:
def validate_clip_perturbations(raw_video_tensor , seed):

    # Reversal Tests 
    reversed_frames = reverse_clip(raw_video_tensor)

    assert not torch.equal(raw_video_tensor, reversed_frames), "Failed: Reversed clip is exactly the same as the original."

    assert torch.equal(raw_video_tensor, reverse_clip(reversed_frames)), "Failed: Double reversal did not match the original."

    ## Shuffling tests
    shuffled_run_1 = shuffle_clip(raw_video_tensor, seed=seed)
    shuffled_run_2 = shuffle_clip(raw_video_tensor, seed=seed)
    
    # Test 1: Determinism 
    assert torch.equal(shuffled_run_1, shuffled_run_2), (
        "Failed Test 1: Shuffling is non-deterministic across runs with the same seed."
    )
    
    # Test 2: Transformation Check
    assert not torch.equal(raw_video_tensor, shuffled_run_1), (
        "Failed Test 2: Shuffled clip is identical to original input."
    )
    
    # Test 3: Conservation of Contents (Set Invariance)
    assert shuffled_run_1.shape == raw_video_tensor.shape, (
        f"Failed Test 3A: Shape mismatch. Expected {raw_video_tensor.shape}, got {shuffled_run_1.shape}"
    )
    
    # B: Value preservation (Sum of all pixel intensities must remain constant)
    assert torch.equal(
        torch.sum(raw_video_tensor.to(torch.int64)), 
        torch.sum(shuffled_run_1.to(torch.int64))
    ), "Failed Test 3B: Pixel sum changed; frames were corrupted or altered during shuffle."


    ### Static tests
    static_clip = make_static_clip(raw_video_tensor)

    # Test 1: Shape Preservation
    assert static_clip.shape == raw_video_tensor.shape, (
        f"Expected shape {raw_video_tensor.shape}, got {static_clip.shape}"
    )
    
    # Test 2: Frame Identity Across Time 
    mid_frame = raw_video_tensor[raw_video_tensor.shape[0] // 2]
    for t in range(static_clip.shape[0]):
        assert torch.equal(static_clip[t], mid_frame), (
            f"Frame at index {t} does not match the reference middle frame."
        )
    
    # Test 3: Transformation Check (Assuming non-static source video)
    assert not torch.equal(raw_video_tensor, static_clip), (
        "Original clip was already identical to static middle frame."
    )
        
    

In [40]:
seed = cfg["perturbation"]["shuffle_seed"]
num_frames = cfg["data"]["num_frames"]
frame_indices = list(range(0, num_frames))

for idx, clip_path in enumerate(clip_paths, 1):
    print(f"Testing clip [{idx}/{len(clip_paths)}]: {clip_path.name}")

    decoder = VideoDecoder(clip_path, device="cuda")
    raw_video_tensor = decoder.get_frames_at(indices=frame_indices).data

    validate_clip_perturbations(raw_video_tensor, seed)

print(f"All {len(clip_paths)}/{len(clip_paths)} clips passed!")

Testing clip [1/30]: -Qz25rXdMjE_000014_000024.mp4
Testing clip [2/30]: -UJgyiWe500_000029_000039.mp4
Testing clip [3/30]: 0S-P4lr_c7s_000022_000032.mp4
Testing clip [4/30]: 27jJKXiB9Y8_000009_000019.mp4
Testing clip [5/30]: 2x1lIrgKxYo_000589_000599.mp4
Testing clip [6/30]: 36E29x22tnQ_000018_000028.mp4
Testing clip [7/30]: 3and4vWkW4s_000011_000021.mp4
Testing clip [8/30]: 3hoSk280ndk_000025_000035.mp4
Testing clip [9/30]: 4hxnP0stMN0_000003_000013.mp4
Testing clip [10/30]: 4wXnaqEDh3c_000014_000024.mp4
Testing clip [11/30]: --dVV4_CSvw_000033_000043.mp4
Testing clip [12/30]: -WH-lxmGJVY_000005_000015.mp4
Testing clip [13/30]: -jOClYqKtE8_000003_000013.mp4
Testing clip [14/30]: 1W7HNDBA4pA_000002_000012.mp4
Testing clip [15/30]: 4JxH3S5JwMs_000003_000013.mp4
Testing clip [16/30]: 5NLQMrXzCQA_000003_000013.mp4
Testing clip [17/30]: 5P0Szq9VDYg_000021_000031.mp4
Testing clip [18/30]: 5Vu8HJ__eMg_000277_000287.mp4
Testing clip [19/30]: 6V6DgC9u7Y0_000000_000010.mp4
Testing clip [20/30]:

In [41]:
dino_checkpoint = cfg["model"]["dino_checkpoint"]
dino_processor = AutoProcessor.from_pretrained(dino_checkpoint , token = token)
dino_model = AutoModel.from_pretrained(dino_checkpoint ,token = token).to("cuda")

dino_model.eval()

# single_frame = raw_video_tensor[0]

# inputs = dino_processor(images = single_frame , return_tensors="pt").to("cuda")

# with torch.no_grad():
#     outputs = dino_model(**inputs)
#     last_hidden_state = outputs.last_hidden_state
# print("Preprocessed Input Shape :", inputs["pixel_values"].shape)
# print("DINOv2 Output Tensor Shape:", last_hidden_state.shape)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

### Why the CLS Token

DINOv2 yields 257 embedding vectors per input frame: a prepended [CLS] token alongside 256 spatial patch tokens. To construct a single frame-level representation without heuristic pooling, we extract the [CLS] token, which is explicitly optimized via DINOv2's image-level objective as a holistic visual descriptor.

Spatial average pooling over the patch tokens remains a viable alternative. However, because both strategies operate strictly over individual 2D spatial manifolds without temporal cross-attention, the resulting embeddings remain inherently permutation-invariant across frames, providing the exact temporal control needed for our baseline.

In [42]:
def encode_dinov2_clip(frames):

    inputs = dino_processor(images = frames , return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = dino_model(**inputs)
        last_hidden_state = outputs.last_hidden_state

    cls_vector = last_hidden_state[:,0,:]

    embedding_avg = torch.mean(cls_vector , dim = 0)

    return embedding_avg
        

In [43]:
emb = encode_dinov2_clip(raw_video_tensor)
print(emb.shape)

torch.Size([768])


In [44]:
a = encode_dinov2_clip(raw_video_tensor)
b = encode_dinov2_clip(shuffle_clip(raw_video_tensor,seed))
print(torch.allclose(a, b, atol=1e-5))

True


In [45]:
a = encode_dinov2_clip(raw_video_tensor)
b = encode_dinov2_clip(make_static_clip(raw_video_tensor))
print(torch.allclose(a, b, atol=1e-5))

False


In [46]:
embeddings = {}

for idx, clip in enumerate(clip_paths, 1):
    print(f"[{idx}/{len(clip_paths)}] {clip.name}")

    decoder = VideoDecoder(clip, device="cuda")
    raw_video_tensor = decoder.get_frames_at(indices=frame_indices).data

    emb         = encode_dinov2_clip(raw_video_tensor)
    reverse_emb = encode_dinov2_clip(reverse_clip(raw_video_tensor))
    shuffle_emb = encode_dinov2_clip(shuffle_clip(raw_video_tensor, seed))
    static_emb  = encode_dinov2_clip(make_static_clip(raw_video_tensor))

    assert torch.allclose(emb, reverse_emb, atol=1e-5), f"reverse failed: {clip.name}"
    assert torch.allclose(emb, shuffle_emb, atol=1e-5), f"shuffle failed: {clip.name}"
    assert not torch.allclose(emb, static_emb, atol=1e-5), f"static failed: {clip.name}"

    embeddings[(clip.name, "original")] = emb.cpu()
    embeddings[(clip.name, "reversed")] = reverse_emb.cpu()
    embeddings[(clip.name, "shuffled")] = shuffle_emb.cpu()
    embeddings[(clip.name, "static")]   = static_emb.cpu()

print(len(embeddings))

[1/30] -Qz25rXdMjE_000014_000024.mp4
[2/30] -UJgyiWe500_000029_000039.mp4
[3/30] 0S-P4lr_c7s_000022_000032.mp4
[4/30] 27jJKXiB9Y8_000009_000019.mp4
[5/30] 2x1lIrgKxYo_000589_000599.mp4
[6/30] 36E29x22tnQ_000018_000028.mp4
[7/30] 3and4vWkW4s_000011_000021.mp4
[8/30] 3hoSk280ndk_000025_000035.mp4
[9/30] 4hxnP0stMN0_000003_000013.mp4
[10/30] 4wXnaqEDh3c_000014_000024.mp4
[11/30] --dVV4_CSvw_000033_000043.mp4
[12/30] -WH-lxmGJVY_000005_000015.mp4
[13/30] -jOClYqKtE8_000003_000013.mp4
[14/30] 1W7HNDBA4pA_000002_000012.mp4
[15/30] 4JxH3S5JwMs_000003_000013.mp4
[16/30] 5NLQMrXzCQA_000003_000013.mp4
[17/30] 5P0Szq9VDYg_000021_000031.mp4
[18/30] 5Vu8HJ__eMg_000277_000287.mp4
[19/30] 6V6DgC9u7Y0_000000_000010.mp4
[20/30] 8TiYnWFX-ow_000042_000052.mp4
[21/30] 01fAWEHzudA_000002_000012.mp4
[22/30] 0oL36GHlSXw_000022_000032.mp4
[23/30] 3sBYgcb4bEY_000003_000013.mp4
[24/30] 4DP5vsyAg1c_000003_000013.mp4
[25/30] 4Zcjoek-1-4_000003_000013.mp4
[26/30] 5RIe5niLskU_000004_000014.mp4
[27/30] 5gVK5JsNRSc_0

In [47]:
def compute_normalized_temporal_distance(original_vec, perturbed_vec, other_original_vecs):
    cos_sim_perturbed = torch.nn.functional.cosine_similarity(original_vec, perturbed_vec, dim=0)
    d_perturbed = 1.0 - cos_sim_perturbed
    d_others = [
    1.0 - torch.nn.functional.cosine_similarity(original_vec, other_vec, dim=0)
    for other_vec in other_original_vecs
    ]
    mean_d_others = torch.stack(d_others).mean()

    return (d_perturbed / mean_d_others).item()
    

In [48]:
import statistics

conditions = ["reversed", "shuffled", "static"]
dino_scores = {c: [] for c in conditions}

clip_names = [p.name for p in clip_paths]

for name in clip_names:
    original_vec = embeddings[(name, "original")]

    other_original_vecs = [
        vec
        for (n, cond), vec in embeddings.items()
        if cond == "original" and n != name
    ]

    for cond in conditions:
        score = compute_normalized_temporal_distance(
            original_vec,
            embeddings[(name, cond)],
            other_original_vecs,
        )
        dino_scores[cond].append(score)

for cond in conditions:
    vals = dino_scores[cond]
    mean_val = statistics.mean(vals)
    median_val = statistics.median(vals)

    if cond in ["reversed", "shuffled"]:
        print(f"{cond:10s} mean={mean_val:.2e}  median={median_val:.2e}")
    else:
        print(f"{cond:10s} mean={mean_val:.4f}  median={median_val:.4f}")

reversed   mean=7.27e-09  median=0.00e+00
shuffled   mean=3.54e-08  median=3.33e-08
static     mean=0.0402  median=0.0301


In [49]:
def encode_vjepa_clips(frames):

    inputs = vjepa_processor(frames , return_tensors= 'pt').to('cuda')

    with torch.no_grad():
        outputs =  vjepa_model(**inputs)
        last_hidden_state =  outputs.last_hidden_state
    vjepa_embedding_mean = torch.mean(last_hidden_state , dim = 1).squeeze(0)
    return vjepa_embedding_mean.to(torch.float32).cpu()
    
    

In [50]:
emd = encode_vjepa_clips(raw_video_tensor)
print(f"{emd.shape}")

torch.Size([1024])


In [51]:
a = encode_vjepa_clips(raw_video_tensor)
b = encode_vjepa_clips(raw_video_tensor)
print(torch.equal(a, b))
print(torch.isnan(a).any())

True
tensor(False)


In [52]:
vjepa_embeddings = {}
for idx , clip in enumerate(clip_paths ,1):
    
    print(f"[{idx}/{len(clip_paths)}] {clip.name}")

    decoder = VideoDecoder(clip, device="cuda")
    raw_video_tensor = decoder.get_frames_at(indices=frame_indices).data

    
    emb         = encode_vjepa_clips(raw_video_tensor)
    reverse_emb = encode_vjepa_clips(reverse_clip(raw_video_tensor))
    shuffle_emb = encode_vjepa_clips(shuffle_clip(raw_video_tensor, seed))
    static_emb  = encode_vjepa_clips(make_static_clip(raw_video_tensor))

    
    vjepa_embeddings[(clip.name, "original")] = emb
    vjepa_embeddings[(clip.name, "reversed")] = reverse_emb
    vjepa_embeddings[(clip.name, "shuffled")] = shuffle_emb
    vjepa_embeddings[(clip.name, "static")]   = static_emb

    
    

[1/30] -Qz25rXdMjE_000014_000024.mp4
[2/30] -UJgyiWe500_000029_000039.mp4
[3/30] 0S-P4lr_c7s_000022_000032.mp4
[4/30] 27jJKXiB9Y8_000009_000019.mp4
[5/30] 2x1lIrgKxYo_000589_000599.mp4
[6/30] 36E29x22tnQ_000018_000028.mp4
[7/30] 3and4vWkW4s_000011_000021.mp4
[8/30] 3hoSk280ndk_000025_000035.mp4
[9/30] 4hxnP0stMN0_000003_000013.mp4
[10/30] 4wXnaqEDh3c_000014_000024.mp4
[11/30] --dVV4_CSvw_000033_000043.mp4
[12/30] -WH-lxmGJVY_000005_000015.mp4
[13/30] -jOClYqKtE8_000003_000013.mp4
[14/30] 1W7HNDBA4pA_000002_000012.mp4
[15/30] 4JxH3S5JwMs_000003_000013.mp4
[16/30] 5NLQMrXzCQA_000003_000013.mp4
[17/30] 5P0Szq9VDYg_000021_000031.mp4
[18/30] 5Vu8HJ__eMg_000277_000287.mp4
[19/30] 6V6DgC9u7Y0_000000_000010.mp4
[20/30] 8TiYnWFX-ow_000042_000052.mp4
[21/30] 01fAWEHzudA_000002_000012.mp4
[22/30] 0oL36GHlSXw_000022_000032.mp4
[23/30] 3sBYgcb4bEY_000003_000013.mp4
[24/30] 4DP5vsyAg1c_000003_000013.mp4
[25/30] 4Zcjoek-1-4_000003_000013.mp4
[26/30] 5RIe5niLskU_000004_000014.mp4
[27/30] 5gVK5JsNRSc_0

In [57]:
torch.save(vjepa_embeddings, "/kaggle/working/vjepa_embeddings.pt")
torch.save(embeddings, "/kaggle/working/dino_embeddings.pt")
print(len(vjepa_embeddings), len(embeddings))

120 120


In [58]:
!ls -la /kaggle/working/*.pt

-rw-r--r-- 1 root root 404287 Sep 16 15:21 /kaggle/working/dino_embeddings.pt
-rw-r--r-- 1 root root 527293 Sep 16 15:21 /kaggle/working/vjepa_embeddings.pt


### Prediction 

I expect V-JEPA's reversed and shuffled scores to be clearly above zero,
since the model has temporal machinery (tubelets, 3D RoPE) that DINOv2
lacks. If instead they land near zero like DINOv2 did, that would mean
the temporal structure isn't visible in the raw embedding geometry.

In [59]:
import statistics

conditions = ["reversed", "shuffled", "static"]
vjepa_scores = {c: [] for c in conditions}

clip_names = [p.name for p in clip_paths]

for name in clip_names:
    original_vec = vjepa_embeddings[(name, "original")]

    other_original_vecs = [
        vec
        for (n, cond), vec in vjepa_embeddings.items()
        if cond == "original" and n != name
    ]

    for cond in conditions:
        score = compute_normalized_temporal_distance(
            original_vec,
            vjepa_embeddings[(name, cond)],
            other_original_vecs,
        )
        vjepa_scores[cond].append(score)

for cond in conditions:
    vals = vjepa_scores[cond]
    mean_val = statistics.mean(vals)
    median_val = statistics.median(vals)

    if cond in ["reversed", "shuffled"]:
        print(f"{cond:10s} mean={mean_val:.6g}  median={median_val:.6g}")
    else:
        print(f"{cond:10s} mean={mean_val:.4f}  median={median_val:.4f}")

reversed   mean=0.0720293  median=0.0579773
shuffled   mean=0.28579  median=0.26121
static     mean=0.4984  median=0.4209


In [60]:
def compute_bootstrap_ci(scores_dict, condition):
    rng = np.random.default_rng(cfg["perturbation"]["bootstrap_seed"])
    n_boot = cfg["perturbation"]["bootstrap_n"]
    scores = np.asarray(scores_dict[condition])

    bootstrap_scores = rng.choice(
        scores,
        size=(n_boot, len(scores)),
        replace=True,
    )
    boot_mean = bootstrap_scores.mean(axis=1)
    ci_low, ci_high = np.percentile(boot_mean, [2.5, 97.5])
    return float(ci_low), float(ci_high)


In [61]:
for model_name, scores_dict in [("V-JEPA 2", vjepa_scores), ("DINOv2", dino_scores)]:
    print(model_name)
    for cond in conditions:
        mean_val = statistics.mean(scores_dict[cond])
        ci_low, ci_high = compute_bootstrap_ci(scores_dict, cond)
        print(f"  {cond:10s} mean={mean_val:.4f}  95% CI [{ci_low:.4f}, {ci_high:.4f}]")
    print()

V-JEPA 2
  reversed   mean=0.0720  95% CI [0.0520, 0.0927]
  shuffled   mean=0.2858  95% CI [0.2405, 0.3335]
  static     mean=0.4984  95% CI [0.4197, 0.5779]

DINOv2
  reversed   mean=0.0000  95% CI [-0.0000, 0.0000]
  shuffled   mean=0.0000  95% CI [-0.0000, 0.0000]
  static     mean=0.0402  95% CI [0.0278, 0.0541]



In [56]:
from scipy.stats import wilcoxon

res_reversed = wilcoxon(vjepa_scores["reversed"] , alternative= "greater")

print(f"Reversed: W-statistic = {res_reversed.statistic:.1f}, p-value = {res_reversed.pvalue:.4e}")

for cond in conditions:
    res = wilcoxon(vjepa_scores[cond], alternative="greater")
    print(f"{cond:10s} W = {res.statistic:6.1f}  p = {res.pvalue:.4e}")


Reversed: W-statistic = 465.0, p-value = 9.3132e-10
reversed   W =  465.0  p = 9.3132e-10
shuffled   W =  465.0  p = 9.3132e-10
static     W =  465.0  p = 9.3132e-10


In [62]:
from scipy.stats import wilcoxon

# Paired signed-rank test: is reversed significantly lower than shuffled?
res_rev_vs_shuf = wilcoxon(
    vjepa_scores["reversed"],
    vjepa_scores["shuffled"],
    alternative="less"
)

print("--- V-JEPA Paired Comparison: Reversed vs. Shuffled ---")
print(f"W-statistic = {res_rev_vs_shuf.statistic:.1f}")
print(f"p-value     = {res_rev_vs_shuf.pvalue:.4e}")

--- V-JEPA Paired Comparison: Reversed vs. Shuffled ---
W-statistic = 0.0
p-value     = 9.3132e-10


## Results

All scores are normalized temporal sensitivity: the cosine distance between a
clip's original and perturbed embedding, divided by that clip's mean cosine
distance to the 29 other clips' originals. A score of 0 means the perturbation
moved the embedding not at all; a score of 1 means it moved it as far as
substituting an unrelated video. n = 30 clips, frozen encoders, mean pooling.
Confidence intervals are percentile bootstrap over clips (10,000 resamples).

| Model | Condition | Mean | 95% CI |
|---|---|---|---|
| V-JEPA 2 | reversed | 0.0717 | [0.0518, 0.0922] |
| V-JEPA 2 | shuffled | 0.2800 | [0.2364, 0.3250] |
| V-JEPA 2 | static | 0.4975 | [0.4192, 0.5757] |
| DINOv2 | reversed | ~0 | [0.0000, 0.0000] |
| DINOv2 | shuffled | ~0 | [0.0000, 0.0000] |
| DINOv2 | static | 0.0397 | [0.0272, 0.0537] |

### Control validation

DINOv2 applied frame-wise and mean-pooled over time is permutation-invariant by
construction, so its reversed and shuffled scores must be zero. Observed values
are on the order of 1e-08, consistent with floating-point accumulation error
alone. Its static score is nonzero (0.0397) because replacing every frame with
the middle frame changes image content, which an appearance model is expected to
register. This confirms the measurement pipeline separates temporal change from
appearance change rather than conflating them.

### V-JEPA 2

Reversal produces a small but consistently nonzero response (CI excludes zero),
so the encoder is not invariant to the direction of time. Shuffling produces a
response roughly four times larger, and the two intervals do not overlap. A
paired Wilcoxon signed-rank test over the same 30 clips gives W = 0
(p = 9.31e-10, one-sided), meaning reversed scored below shuffled on every clip
without exception.

One-sample Wilcoxon tests against zero return W = 465 and p = 9.31e-10 for all
three conditions. W = 465 is the maximum attainable at n = 30, and the p-value
is the 2^-30 floor, so these tests establish that every clip scored above zero
but cannot discriminate between conditions.

### Interpretation and limits

Under this metric, V-JEPA 2's representation is markedly more sensitive to
disruption of motion continuity than to inversion of temporal direction.

Three limits constrain how far this generalizes:

1. The shuffled condition is confounded. Random frame order destroys temporal
   structure but also introduces jump cuts, altering low-level frame-to-frame
   statistics. The reversed-versus-shuffled gap cannot be attributed to
   temporal structure alone.
2. Clips are drawn from Kinetics-mini (archery, bowling, high jump), a set
   where actions are often identifiable from appearance. Something-Something v2
   would test motion dependence more strictly.
3. This measures raw embedding geometry with nothing trained on top. It does
   not speak to what a trained probe could extract from the same features.